In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
import matplotlib.pyplot as plt
import os
from datetime import datetime

# Directory where the CSV files are stored
directory_path = 'data'

# Delete old merged_result.csv file
if os.path.exists(directory_path + '/merged_result.csv'):
    os.remove(directory_path + '/merged_result.csv')
    print(" ### Cleared merged_result.csv file ### \n")

# Merge all CSV files in the directory
def merge_all_csv_files(directory):
    all_files = [file for file in os.listdir(directory) if file.endswith('.csv')]
    merged_df = pd.concat([pd.read_csv(os.path.join(directory, file), sep=';') for file in all_files])
    return merged_df

df = merge_all_csv_files(directory_path)

# Role = User
df = df[df['Role'] == 'User']

# Filter data frame to let columns: Id, Role, Message, Datetime, Classification and Topic
df = df[['Id', 'Message', 'Datetime', 'Classification', 'Topic']]

# eliminar linhas onde a coluna topic é nula
df = df[df['Topic'].notna()]
# eliminar linhas onde a coluna classification é nula
df = df[df['Classification'].notna()]

# salvando o DataFrame filtrado em um novo arquivo CSV
df.to_csv('data/temp/statistics.csv', index=False, sep=';')

# Função para converter o formato específico de data
def convert_datetime(datetime_str):
    try:
        # Primeiro tenta o formato original
        parts = datetime_str.strip().split(' - ')
        if len(parts) != 2:
            return pd.NaT

        time_str, date_str = parts

        # Remove espaços extras
        time_str = time_str.strip()
        date_str = date_str.strip()

        # Verifica se o formato está correto
        if not (len(time_str) == 8 and len(date_str) == 10):
            return pd.NaT

        # Tenta converter para datetime
        dt = datetime.strptime(f'{date_str} {time_str}', '%d/%m/%Y %H:%M:%S')
        return pd.Timestamp(dt)
    except Exception as e:
        return pd.NaT

# Converter Datetime para datetime
df['Datetime'] = df['Datetime'].apply(convert_datetime)

# Remover linhas com datas inválidas
df = df.dropna(subset=['Datetime'])

# Ordenar por Id e Datetime para manter a ordem correta das conversas
df = df.sort_values(['Id', 'Datetime'])

# Criar uma coluna com o número da mensagem dentro da sessão
df['MessageOrder'] = df.groupby('Id').cumcount() + 1

# Preencher valores nulos
df['Topic'] = df['Topic'].fillna('No Topic')
df['Classification'] = df['Classification'].fillna('Uncategorized')

# print("\nExemplo de uma sessão de conversa ordenada:")
# print(df.groupby('Id').head(1))

# Agregando as classificações e tópicos por sessão mantendo a ordem
df['ClassificationSequence'] = df.groupby('Id')['Classification'].transform(lambda x: ' -> '.join(x))
df['TopicSequence'] = df.groupby('Id')['Topic'].transform(lambda x: ' -> '.join(x))

# Removendo duplicatas para ter uma linha por sessão
df_sequences = df.drop_duplicates('Id')[['Id', 'ClassificationSequence', 'TopicSequence']]

# print("\nExemplo de sequências de classificação em uma sessão:")
# print(df_sequences[['ClassificationSequence', 'TopicSequence']].head())

# Criar dummies para as sequências
df_bin = pd.get_dummies(df_sequences.drop('Id', axis=1))

# Verificando a estrutura após one-hot encoding
# print("\nColunas após one-hot encoding:")
# print(df_bin.columns.tolist())

# Ajustando parâmetros do Apriori para valores mais baixos
frequent_itemsets = apriori(df_bin,
                          min_support=0.05,  # Reduzindo o suporte mínimo
                          use_colnames=True)

print("\nNúmero de itemsets frequentes encontrados:", len(frequent_itemsets))

# Gerando regras com parâmetros mais flexíveis
rules = association_rules(frequent_itemsets,
                         metric="confidence",
                         min_threshold=0.7)  # Reduzindo a confiança mínima

# Exibindo métricas principais
if len(rules) > 0:
    print("\nRegras de associação encontradas:")
    print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

    # # Visualização das regras mais fortes
    # plt.figure(figsize=(10, 6))
    # plt.scatter(rules['support'], rules['confidence'], alpha=0.5)
    # plt.xlabel('Suporte')
    # plt.ylabel('Confiança')
    # plt.title('Suporte vs Confiança para Regras de Associação')
    # plt.show()
else:
    print("\nNenhuma regra de associação encontrada com os parâmetros atuais.")




Número de itemsets frequentes encontrados: 4

Regras de associação encontradas:
            antecedents                                    consequents  \
0  (TopicSequence_Java)  (ClassificationSequence_Conceptual Questions)   

    support  confidence      lift  
0  0.057143         1.0  7.777778  
